In [3]:
import pandas as pd
import numpy as np

# Load datasets
results_df = pd.read_csv('data/results_clean.csv')
shootouts_df = pd.read_csv('data/shootouts_clean.csv')
goalscorers_df = pd.read_csv('data/goalscorers_clean.csv')

display(results_df.head())
display(shootouts_df.head())
display(goalscorers_df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id
0,1956-09-23,Germany,Netherlands,2,1,Friendly,Essen,Germany,False,1956-09-23_Germany_Netherlands
1,1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany,False,1957-07-28_Germany_England
2,1957-10-13,Germany,Netherlands,2,0,Friendly,Berlin,Germany,False,1957-10-13_Germany_Netherlands
3,1957-11-03,Netherlands,Austria,8,1,European Championship,Berlin,Germany,True,1957-11-03_Netherlands_Austria
4,1957-11-03,Germany,England,0,4,European Championship,Berlin,Germany,False,1957-11-03_Germany_England


,date,home_team,away_team,winner,first_shooter,match_id
0,1984-05-27,England,Sweden,Sweden,NaN,1984-05-27_England_Sweden
1,1988-06-12,China PR,Brazil,Brazil,Brazil,1988-06-12_China_PR_Brazil
2,1995-06-13,Sweden,China PR,China PR,China PR,1995-06-13_Sweden_China_PR
3,1996-05-18,Japan,Canada,Japan,NaN,1996-05-18_Japan_Canada
4,1998-07-25,China PR,Norway,China PR,NaN,1998-07-25_China_PR_Norway


,date,home_team,away_team,team,scorer,minute,own_goal,penalty,match_id
0,1984-04-08,England,Denmark,England,Linda Curl,31,False,False,1984-04-08_England_Denmark
1,1984-04-08,England,Denmark,Denmark,Inge Hindkjær,49,False,True,1984-04-08_England_Denmark
2,1984-04-08,England,Denmark,England,Elisabeth Deighan,51,False,False,1984-04-08_England_Denmark
3,1984-04-08,Italy,Sweden,Italy,Carolina Morace,18,False,False,1984-04-08_Italy_Sweden
4,1984-04-08,Italy,Sweden,Sweden,Helen Johansson,23,False,False,1984-04-08_Italy_Sweden


### Elo Ratings

An ELO rating for football teams is a numerical system that measures a team's relative skill level, calculated by adding or subtracting points based on match results, with stronger teams gaining fewer points for a win and losing more for a loss. Websites like [World Football Elo Ratings](https://www.eloratings.net/) and [Club Elo Ratings](http://clubelo.com/) provide real-time ELO rankings for national and club teams, respectively, showing how teams are performing against one another. 

#### How ELO Works
- **Based on Performance:** The ELO system is dynamic and updates with each match played, adjusting a team's rating based on the outcome and the relative strength of their opponent. 
- **Points Exchange:** When a team wins, they gain points, and when they lose, they lose points. 
- **Expected vs. Actual Result:** The amount of points exchanged depends on the difference in ratings between the two teams; a higher-rated team is expected to win, so they gain fewer points for a win and lose more for a loss compared to a lower-rated team. 
- **Draws and Penalties:** A draw results in a small number of points being exchanged, while a match won on penalties is also considered a draw for ELO calculation purposes, according to the [Wikipedia World Football Elo Ratings page](https://en.wikipedia.org/wiki/World_Football_Elo_Ratings).

#### Examples of Football ELO Ratings
- **National Teams:** World Football Elo Ratings provides current ELO ratings for national teams, showing the highest-rated teams based on their match results. 
- **Club Teams:** Websites such as [EloFootball.com](https://elofootball.com/) and Club Elo Ratings maintain ELO rankings specifically for club teams across various leagues. 
- **FIFA/Coca-Cola Ranking:** The official FIFA/Coca-Cola World Ranking uses the ELO model to assess the strength of national teams. 

In [4]:
import numpy as np

# Parameters
K = 40  # can tune this later

# Initialize Elo dictionary
elo_ratings = {}

def get_elo(team):
    return elo_ratings.get(team, 1500)  # default starting Elo

def update_elo(team_a, team_b, score_a, score_b):
    R_a = get_elo(team_a)
    R_b = get_elo(team_b)
    
    # Expected outcome
    E_a = 1 / (1 + 10 ** ((R_b - R_a) / 400))
    E_b = 1 - E_a
    
    # Actual outcome
    if score_a > score_b:
        S_a, S_b = 1, 0
    elif score_a < score_b:
        S_a, S_b = 0, 1
    else:
        S_a, S_b = 0.5, 0.5

    # Update
    elo_ratings[team_a] = R_a + K * (S_a - E_a)
    elo_ratings[team_b] = R_b + K * (S_b - E_b)

# Sort matches chronologically
results_df = results_df.sort_values('date').reset_index(drop=True)

# Add columns for pre-match Elo
results_df['elo_home'] = np.nan
results_df['elo_away'] = np.nan

# Loop through matches
for i, row in results_df.iterrows():
    home, away = row['home_team'], row['away_team']
    h_score, a_score = row['home_score'], row['away_score']
    
    # Pre-match Elo
    results_df.at[i, 'elo_home'] = get_elo(home)
    results_df.at[i, 'elo_away'] = get_elo(away)
    
    # Post-match update
    update_elo(home, away, h_score, a_score)

results_df['elo_diff'] = results_df['elo_home'] - results_df['elo_away']

display(results_df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id,elo_home,elo_away,elo_diff
0,1956-09-23,Germany,Netherlands,2,1,Friendly,Essen,Germany,False,1956-09-23_Germany_Netherlands,1500.000000,1500.000000,0.000000
1,1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany,False,1957-07-28_Germany_England,1520.000000,1500.000000,20.000000
2,1957-10-13,Germany,Netherlands,2,0,Friendly,Berlin,Germany,False,1957-10-13_Germany_Netherlands,1518.849977,1480.000000,38.849977
3,1957-11-03,Netherlands,Austria,8,1,European Championship,Berlin,Germany,True,1957-11-03_Netherlands_Austria,1462.227110,1500.000000,-37.772890
4,1957-11-03,Germany,England,0,4,European Championship,Berlin,Germany,False,1957-11-03_Germany_England,1536.622867,1501.150023,35.472845


### Rolling Performance features

In [5]:
N = 5

def rolling_team_stats(df, team_col, goals_for_col, goals_against_col, N):
    stats = []
    team_history = {}
    
    for _, row in df.iterrows():
        team_home = row['home_team']
        team_away = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']
        
        # Home team features
        prev_home = team_history.get(team_home, [])
        if prev_home:
            gf = np.mean([x[0] for x in prev_home[-N:]])
            ga = np.mean([x[1] for x in prev_home[-N:]])
        else:
            gf, ga = 0, 0
        row_home = {'match_id': row['match_id'], 'team': team_home, 'gf_roll': gf, 'ga_roll': ga}
        stats.append(row_home)
        
        # Away team features
        prev_away = team_history.get(team_away, [])
        if prev_away:
            gf = np.mean([x[0] for x in prev_away[-N:]])
            ga = np.mean([x[1] for x in prev_away[-N:]])
        else:
            gf, ga = 0, 0
        row_away = {'match_id': row['match_id'], 'team': team_away, 'gf_roll': gf, 'ga_roll': ga}
        stats.append(row_away)
        
        # Update history
        team_history.setdefault(team_home, []).append((home_goals, away_goals))
        team_history.setdefault(team_away, []).append((away_goals, home_goals))
    
    return pd.DataFrame(stats)

rolling_stats = rolling_team_stats(results_df, 'team', 'gf_roll', 'ga_roll', N)

display(rolling_stats)

,match_id,team,gf_roll,ga_roll
0,1956-09-23_Germany_Netherlands,Germany,0.0,0.0
1,1956-09-23_Germany_Netherlands,Netherlands,0.0,0.0
2,1957-07-28_Germany_England,Germany,2.0,1.0
3,1957-07-28_Germany_England,England,0.0,0.0
4,1957-10-13_Germany_Netherlands,Germany,1.5,1.0
...,...,...,...,...
21145,2025-08-13_Philippines_Myanmar,Myanmar,3.0,0.6
21146,2025-08-16_Myanmar_Thailand,Myanmar,2.6,0.6
21147,2025-08-16_Myanmar_Thailand,Thailand,3.4,0.2
21148,2025-08-19_Vietnam_Thailand,Vietnam,3.2,0.4


In [6]:
results_df = results_df.merge(
    rolling_stats[['match_id', 'gf_roll', 'ga_roll']],
    on='match_id',
    how='right'
)

results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id,elo_home,elo_away,elo_diff,gf_roll,ga_roll
0,1956-09-23,Germany,Netherlands,2,1,Friendly,Essen,Germany,False,1956-09-23_Germany_Netherlands,1500.000000,1500.0,0.000000,0.0,0.0
1,1956-09-23,Germany,Netherlands,2,1,Friendly,Essen,Germany,False,1956-09-23_Germany_Netherlands,1500.000000,1500.0,0.000000,0.0,0.0
2,1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany,False,1957-07-28_Germany_England,1520.000000,1500.0,20.000000,2.0,1.0
3,1957-07-28,Germany,England,1,1,Friendly,Stuttgart,Germany,False,1957-07-28_Germany_England,1520.000000,1500.0,20.000000,0.0,0.0
4,1957-10-13,Germany,Netherlands,2,0,Friendly,Berlin,Germany,False,1957-10-13_Germany_Netherlands,1518.849977,1480.0,38.849977,1.5,1.0


### Head-to-head stats

In [8]:
def compute_h2h_features_single_row(df):
    h2h_history = {}
    features = []

    for _, row in df.iterrows():
        home, away = row['home_team'], row['away_team']
        key = tuple(sorted([home, away]))
        past = h2h_history.get(key, [])

        # Count previous results
        home_wins = sum(1 for r in past if r == home)
        away_wins = sum(1 for r in past if r == away)
        draws = sum(1 for r in past if r == 'draw')
        total = len(past)

        if total > 0:
            home_winrate = home_wins / total
            away_winrate = away_wins / total
            draw_rate = draws / total
        else:
            # default neutral values
            home_winrate = 0.5
            away_winrate = 0.5
            draw_rate = 0.0

        features.append({
            'match_id': row['match_id'],
            'home_h2h_winrate': home_winrate,
            'away_h2h_winrate': away_winrate,
            'h2h_draw_rate': draw_rate
        })

        # Record current result for future matches
        if row['home_score'] > row['away_score']:
            result = home
        elif row['home_score'] < row['away_score']:
            result = away
        else:
            result = 'draw'

        h2h_history.setdefault(key, []).append(result)

    return pd.DataFrame(features)

h2h_df = compute_h2h_features_single_row(results_df)
h2h_df.head()

,match_id,home_h2h_winrate,away_h2h_winrate,h2h_draw_rate
0,1956-09-23_Germany_Netherlands,0.5,0.5,0.0
1,1956-09-23_Germany_Netherlands,1.0,0.0,0.0
2,1957-07-28_Germany_England,0.5,0.5,0.0
3,1957-07-28_Germany_England,0.0,0.0,1.0
4,1957-10-13_Germany_Netherlands,1.0,0.0,0.0
